# Extended Data Figure 8: Reputation geometry along the ALLD–DISC edge

This notebook computes every self-consistent reputation equilibrium of the private-assessment closure along the
ALLD–DISC edge of the strategy simplex, classifies its stability, and identifies the branch selected by the main
convention (relaxation from all-good reputations at each composition). It then evaluates the payoff difference
between DISC and ALLD on the selected branch. Run from this directory; set `RECOMPUTE=True` to regenerate the data.

Parameters: $b/c=5$, $e_1=e_2=0.02$; without inertia $(k_1,k_2)=(0,0)$; full one-step inertia $(1,1)$ with
$\alpha=\beta=1$; selected multi-step inertia SC $(2,0)$, SH $(3,0)$, SJ $(4,3)$, SS $(1,2)$ with $\alpha=\beta=1$.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch

ROOT = Path.cwd()
DATA = ROOT / 'data' / 'extended_data_figure8'
FIGURES = ROOT / 'figures' / 'extended_data_figure8'
DATA.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
RECOMPUTE = False

E1 = E2 = 0.02
B_OVER_C = 5.0
NORMS = {'SC': (1., 0.), 'SH': (0., 0.), 'SJ': (0., 1.), 'SS': (1., 1.)}
NAMES = {'SC': 'Scoring', 'SH': 'Shunning', 'SJ': 'Stern judging', 'SS': 'Simple standing'}
SELECTED = {'SC': (2, 0), 'SH': (3, 0), 'SJ': (4, 3), 'SS': (1, 2)}
CONDITIONS = ('without', 'one', 'multi')
COLORS = {'without': '#6B6B6B', 'one': '#0077BB', 'multi': '#D55E00'}
LABELS = {'without': 'Without inertia', 'one': 'One-step inertia', 'multi': 'Multi-step inertia'}
X = np.r_[np.linspace(0.01, 0.98, 971), np.linspace(0.9802, 1.0, 100)]   # f_DISC grid on the ALLD-DISC edge (f_ALLC = 0)
N_RBAR = 8001                                                             # resolution of the equilibrium scan
CONFIG = dict(e1=E1, e2=E2, b_over_c=B_OVER_C, selected=SELECTED, n_points=len(X), n_rbar=N_RBAR)

def architecture(norm, condition):
    if condition == 'without': return (0, 0, 0.0, 0.0)
    if condition == 'one': return (1, 1, 1.0, 1.0)
    return (*SELECTED[norm], 1.0, 1.0)

## Stationary response and self-consistency

$\mathcal R(u)$ is the stationary good-readout probability of the direct-reset record for independent assessments
that are positive with probability $u$. On the ALLD–DISC edge the self-consistency problem reduces exactly to one
equation in the population mean reputation $\bar r$: given $\bar r$, $r_{\rm ALLD}=\mathcal R(\varphi_{\rm ALLD}(\bar r))$
and $r_{\rm DISC}=(\bar r-f_{\rm ALLD}r_{\rm ALLD})/f_{\rm DISC}$. All roots are located on a fine $\bar r$ grid and
refined by bisection; stability is that of the auxiliary flow $\mathrm dr_I/\mathrm d\tau=\mathcal R(\varphi_I)-r_I$.

In [2]:
def response(u, k1, k2, alpha, beta):
    u = np.clip(np.asarray(u, float), 0.0, 1.0); v = 1.0 - u
    Z = v ** k1 + u ** k2 - v ** k1 * u ** k2
    warning = sum((alpha * v) ** m for m in range(1, k1 + 1)) if k1 else 0.0 * u
    recovery = sum(u ** n * (1 - beta ** n) for n in range(1, k2 + 1)) if k2 else 0.0 * u
    return (u ** (k2 + 1) * (1 + warning) + v ** (k1 + 1) * recovery) / Z

def assessment(norm):
    p, q = NORMS[norm]; eps = (1 - E1) * (1 - E2) + E1 * E2
    return np.array([eps, p * (eps - E2) + q * (1 - eps - E2) + E2, E2, q * (1 - 2 * E2) + E2])

def flow(r, f, A, arch):
    g = (f * r).sum(-1); g2 = (f * r * r).sum(-1)
    u = np.stack([g * A[0] + (1 - g) * A[1], g * A[2] + (1 - g) * A[3],
                  g2 * A[0] + (g - g2) * (A[1] + A[2]) + (1 - 2 * g + g2) * A[3]], -1)
    return response(u, *arch) - r

def jacobian(r, f, A, arch, h=1e-7):
    J = np.empty(r.shape + (3,))
    for j in range(3):
        e = np.zeros(3); e[j] = h
        J[..., j] = (flow(np.clip(r + e, 0, 1), f, A, arch) - flow(np.clip(r - e, 0, 1), f, A, arch)) / (2 * h)
    return J

def edge_equilibria(norm, condition):
    A = assessment(norm); arch = architecture(norm, condition)
    grid = np.linspace(0.0, 1.0, N_RBAR); rows = []
    fD = (1 - X)[:, None]; fR = X[:, None]
    def F(gb, fD, fR):
        rD = response(gb * A[2] + (1 - gb) * A[3], *arch)
        rC = response(gb * A[0] + (1 - gb) * A[1], *arch)
        rR = (gb - fD * rD) / fR
        g2 = fD * rD ** 2 + fR * rR ** 2
        val = response(g2 * A[0] + (gb - g2) * (A[1] + A[2]) + (1 - 2 * gb + g2) * A[3], *arch) - rR
        return np.where((rR > -0.2) & (rR < 1.2), val, np.nan), rC, rD, rR
    val = F(grid[None, :], fD, fR)[0]
    s = np.sign(val); ii, jj = np.nonzero((s[:, :-1] * s[:, 1:] < 0) | (val[:, :-1] == 0))
    lo, hi = grid[jj].copy(), grid[jj + 1].copy(); vlo = F(lo, fD[ii, 0], fR[ii, 0])[0]
    for _ in range(60):
        mid = 0.5 * (lo + hi); vm = F(mid, fD[ii, 0], fR[ii, 0])[0]; left = np.sign(vm) == np.sign(vlo)
        lo = np.where(left, mid, lo); vlo = np.where(left, vm, vlo); hi = np.where(left, hi, mid)
    gb = 0.5 * (lo + hi); _, rC, rD, rR = F(gb, fD[ii, 0], fR[ii, 0])
    keep = (rR >= -1e-9) & (rR <= 1 + 1e-9)
    ii, gb, rC, rD, rR = ii[keep], gb[keep], rC[keep], rD[keep], np.clip(rR[keep], 0, 1)
    r = np.stack([rC, rD, rR], -1); f = np.stack([0 * gb, 1 - X[ii], X[ii]], -1)
    eig = np.linalg.eigvals(jacobian(r, f, A, arch)).real.max(-1)
    residual = np.abs(flow(r, f, A, arch)).max(-1)
    return pd.DataFrame(dict(norm=norm, condition=condition, f_DISC=X[ii], r_ALLC=rC, r_ALLD=rD, r_DISC=rR,
                             max_eigenvalue=eig, stable=eig < -1e-9, residual=residual))

def all_good_branch(norm, condition, tol=1e-12):
    """Main convention: relaxation from all-good reputations at each composition (RK4, step 0.25)."""
    A = assessment(norm); arch = architecture(norm, condition)
    f = np.stack([0 * X, 1 - X, X], -1); r = np.ones_like(f); active = np.ones(len(X), bool)
    for _ in range(2_000_000):
        idx = np.flatnonzero(active)
        if idx.size == 0: break
        rr, ff = r[idx], f[idx]; k1 = flow(rr, ff, A, arch)
        done = np.abs(k1).max(-1) <= tol; active[idx[done]] = False
        rr, ff, k1, idx = rr[~done], ff[~done], k1[~done], idx[~done]
        if idx.size == 0: continue
        k2 = flow(np.clip(rr + .125 * k1, 0, 1), ff, A, arch); k3 = flow(np.clip(rr + .125 * k2, 0, 1), ff, A, arch)
        k4 = flow(np.clip(rr + .25 * k3, 0, 1), ff, A, arch)
        r[idx] = np.clip(rr + .25 * (k1 + 2 * k2 + 2 * k3 + k4) / 6, 0, 1)
    residual = np.abs(flow(r, f, A, arch)).max(-1)
    gbar = (f * r).sum(-1)
    payoff_gap = B_OVER_C * X * (r[:, 2] - r[:, 1]) - gbar          # (Pi_DISC - Pi_ALLD) / [c (1 - e1)]
    return pd.DataFrame(dict(norm=norm, condition=condition, f_DISC=X, r_ALLC=r[:, 0], r_ALLD=r[:, 1],
                             r_DISC=r[:, 2], payoff_gap=payoff_gap, residual=residual))

## Compute or load data

In [3]:
equilibria_file, selected_file, config_file = DATA / 'edge_equilibria.csv', DATA / 'selected_branch.csv', DATA / 'input_config.json'
if RECOMPUTE:
    equilibria = pd.concat([edge_equilibria(n, c) for n in NORMS for c in CONDITIONS], ignore_index=True)
    selected = pd.concat([all_good_branch(n, c) for n in NORMS for c in CONDITIONS], ignore_index=True)
    equilibria.to_csv(equilibria_file, index=False); selected.to_csv(selected_file, index=False)
    config_file.write_text(json.dumps(CONFIG, indent=2) + '\n')
else:
    if json.loads(config_file.read_text()) != json.loads(json.dumps(CONFIG)):
        raise ValueError('Saved configuration differs; set RECOMPUTE=True.')
    equilibria = pd.read_csv(equilibria_file); selected = pd.read_csv(selected_file)
assert equilibria.residual.max() < 1e-10 and selected.residual.max() <= 1e-12

# Folds (saddle-node points): a stable and an unstable root appear or disappear together between adjacent grid points.
# The fold is located at the grid point where the pair still exists, at the mean of the two colliding roots.
def folds(eq):
    out = []
    for (n, c), g in eq.groupby(['norm', 'condition'], sort=False):
        roots = {x: h for x, h in g.groupby('f_DISC')}
        empty = g.iloc[:0]
        for x0, x1 in zip(X[:-1], X[1:]):
            h0, h1 = roots.get(x0, empty), roots.get(x1, empty)
            ds, du = h1.stable.sum() - h0.stable.sum(), (~h1.stable).sum() - (~h0.stable).sum()
            if ds == 0 or ds != du: continue
            x, h = (x0, h0) if ds < 0 else (x1, h1)
            rs, ru = h[h.stable].r_DISC.to_numpy(), h[~h.stable].r_DISC.to_numpy()
            d = np.abs(rs[:, None] - ru[None, :]); i, j = np.unravel_index(d.argmin(), d.shape)
            out.append(dict(norm=n, condition=c, f_DISC=x, r_DISC=0.5 * (rs[i] + ru[j]), root_separation=d[i, j]))
    return pd.DataFrame(out)

fold_table = folds(equilibria)
# Discontinuities of the selected branch and zero crossings of the payoff gap
events = []
for (n, c), g in selected.groupby(['norm', 'condition'], sort=False):
    r = g.r_DISC.to_numpy(); d = g.payoff_gap.to_numpy(); x = g.f_DISC.to_numpy()
    for i in np.flatnonzero(np.sign(d[:-1]) * np.sign(d[1:]) < 0):
        jump = abs(r[i + 1] - r[i])
        events.append(dict(norm=n, condition=c, f_DISC=0.5 * (x[i] + x[i + 1]), r_DISC_jump=jump,
                           kind='discontinuous (fold)' if jump > 0.05 else 'continuous (edge equilibrium)'))
events = pd.DataFrame(events)
events.to_csv(DATA / 'payoff_sign_changes.csv', index=False); fold_table.to_csv(DATA / 'folds.csv', index=False)
print(events.to_string(index=False))

norm condition  f_DISC  r_DISC_jump                          kind
  SC   without  0.2125     0.000028 continuous (edge equilibrium)
  SC       one  0.8955     0.858309          discontinuous (fold)
  SC     multi  0.0795     0.000534 continuous (edge equilibrium)
  SH       one  0.9715     0.968545          discontinuous (fold)
  SH     multi  0.5105     0.674753          discontinuous (fold)
  SJ       one  0.7745     0.465116          discontinuous (fold)
  SJ     multi  0.2825     0.002742 continuous (edge equilibrium)
  SS   without  0.3565     0.000084 continuous (edge equilibrium)
  SS       one  0.2385     0.000201 continuous (edge equilibrium)
  SS     multi  0.2065     0.000294 continuous (edge equilibrium)


## Figure

Upper row: self-consistent DISC reputations along the ALLD–DISC edge (solid: stable; dashed: unstable; diamonds:
folds). Lower row: normalized payoff difference $(\Pi_{\rm DISC}-\Pi_{\rm ALLD})/[c(1-e_1)]$ on the branch selected
from all-good reputations. Circles mark continuous sign changes (edge equilibria); open triangles mark sign changes
produced by loss of the high-reputation branch at a fold.

In [4]:
from matplotlib import font_manager
INSTALLED_FONTS = {f.name for f in font_manager.fontManager.ttflist}
SANS = next((n for n in ('Helvetica', 'TeX Gyre Heros', 'Arial') if n in INSTALLED_FONTS), 'DejaVu Sans')
mpl.rcParams.update({'font.family': SANS, 'mathtext.fontset': 'stix', 'font.size': 9,
                     'axes.labelsize': 9, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'axes.linewidth': 0.8,
                     'axes.spines.top': False, 'axes.spines.right': False, 'pdf.fonttype': 42, 'svg.fonttype': 'none'})
NAVY = '#1F3864'

def branch_lines(ax, g, color):
    """Draw stable (solid) and unstable (dashed) roots, joining points that continue each other in (f_DISC, r_DISC)."""
    for stable, style, lw in [(True, '-', 1.8), (False, (0, (2.4, 1.6)), 1.1)]:
        pts = g[g.stable == stable].sort_values(['f_DISC', 'r_DISC'])
        tracks = []
        for x, grp in pts.groupby('f_DISC', sort=True):
            for r in grp.r_DISC:
                cands = [t for t in tracks if t[-1][0] < x and x - t[-1][0] < 0.012 and abs(t[-1][1] - r) < 0.06]
                if cands: min(cands, key=lambda t: abs(t[-1][1] - r)).append((x, r))
                else: tracks.append([(x, r)])
        for t in tracks:
            if len(t) > 1:
                t = np.array(t); ax.plot(t[:, 0], t[:, 1], color=color, lw=lw, ls=style, solid_capstyle='round', zorder=3)

def decorate(ax, letter, title=None):
    ax.grid(color='#D9D9D9', lw=0.45, alpha=0.45, zorder=0)
    ax.text(-0.25, 1.09, letter, transform=ax.transAxes, fontsize=14, va='bottom', ha='left')
    if title: ax.set_title(title, fontsize=10.5, pad=17)

fig = plt.figure(figsize=(7.5, 4.9))
grid = fig.add_gridspec(2, 4, left=0.085, right=0.985, top=0.89, bottom=0.215, wspace=0.36, hspace=0.50)
for j, norm in enumerate(NORMS):
    ax = fig.add_subplot(grid[0, j]); decorate(ax, 'abcd'[j], NAMES[norm])
    for c in CONDITIONS:
        branch_lines(ax, equilibria[(equilibria.norm == norm) & (equilibria.condition == c)], COLORS[c])
        ft = fold_table[(fold_table.norm == norm) & (fold_table.condition == c)] if len(fold_table) else fold_table
        if len(ft): ax.plot(ft.f_DISC, ft.r_DISC, 'D', ms=4.6, mfc='white', mec=COLORS[c], mew=1.2, zorder=5)
    ax.set(xlim=(0, 1), ylim=(-0.03, 1.03), xticks=[0, 0.5, 1], yticks=[0, 0.5, 1])
    k1, k2 = SELECTED[norm]
    ax.text(1.0, 1.015, rf'$(k_1,k_2)=({k1},{k2})$', transform=ax.transAxes, ha='right', va='bottom', fontsize=8.5,
            color=COLORS['multi'])
    if j == 0: ax.set_ylabel(r'DISC reputation, $r_{\rm DISC}$')
    ax2 = fig.add_subplot(grid[1, j]); decorate(ax2, 'efgh'[j])
    ax2.axhline(0, color='#303030', lw=0.8, zorder=2)
    for c in CONDITIONS:
        g = selected[(selected.norm == norm) & (selected.condition == c)]
        x, d, r = g.f_DISC.to_numpy(), g.payoff_gap.to_numpy(), g.r_DISC.to_numpy()
        breaks = np.flatnonzero(np.abs(np.diff(r)) > 0.05) + 1
        for seg_x, seg_d in zip(np.split(x, breaks), np.split(d, breaks)):
            ax2.plot(seg_x, seg_d, color=COLORS[c], lw=1.8, zorder=3)
        ev = events[(events.norm == norm) & (events.condition == c)] if len(events) else events
        for _, e in ev.iterrows():
            if e.kind.startswith('discontinuous'):
                ax2.plot(e.f_DISC, 0, marker='^', ms=5.6, mfc='white', mec=COLORS[c], mew=1.2, zorder=6)
            else:
                ax2.plot(e.f_DISC, 0, marker='o', ms=4.8, mfc=COLORS[c], mec='white', mew=0.6, zorder=6)
    ax2.set(xlim=(0, 1), xticks=[0, 0.5, 1], ylim=(-1.0, 4.2), yticks=[-1, 0, 1, 2, 3, 4])
    ax2.set_xlabel(r'DISC frequency, $f_{\rm DISC}$')
    if j == 0: ax2.set_ylabel('DISC payoff advantage')
handles = [Line2D([0], [0], color=COLORS[c], lw=1.8, label=LABELS[c]) for c in CONDITIONS]
handles += [Line2D([0], [0], color='#222222', lw=1.8, label='Stable equilibrium'),
            Line2D([0], [0], color='#222222', lw=1.1, ls=(0, (2.4, 1.6)), label='Unstable equilibrium'),
            Line2D([0], [0], marker='D', ms=4.8, mfc='white', mec='#222222', mew=1.2, ls='None', label='Fold'),
            Line2D([0], [0], marker='o', ms=5, mfc='#222222', mec='white', ls='None', label='Edge equilibrium'),
            Line2D([0], [0], marker='^', ms=5.6, mfc='white', mec='#222222', mew=1.2, ls='None', label='Sign change at fold')]
leg = fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.535, 0.01), ncol=4, frameon=True, fancybox=False,
                 fontsize=8.5, columnspacing=1.8, handlelength=2.2, borderpad=0.7)
leg.get_frame().set_edgecolor(NAVY); leg.get_frame().set_linewidth(0.9)
for ext in ('pdf', 'svg'):
    fig.savefig(FIGURES / f'Extended_Data_Figure8.{ext}', facecolor='white')
fig.savefig(FIGURES / 'Extended_Data_Figure8.png', dpi=300, facecolor='white')
plt.show(); plt.close(fig)
print('Saved Extended Data Figure 8 (PDF, SVG and PNG).')

Saved Extended Data Figure 8 (PDF, SVG and PNG).
